# `mutate` — Reference

`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.

### Calling Styles

1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **External file specification**: `.pt.mutate("@features.txt")`

---


In [18]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [19]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
299,Gentoo,5950.0,45.2,2.912327
245,Gentoo,5100.0,46.1,2.399763
4,Adelie,3450.0,36.7,2.561456
22,Adelie,3800.0,35.9,2.948456
180,Chinstrap,3700.0,46.4,1.718564
44,Adelie,3000.0,37.0,2.191381
339,Gentoo,NaN,NaN,NaN
267,Gentoo,5400.0,50.5,2.117439
211,Chinstrap,3525.0,45.6,1.695233
52,Adelie,3450.0,35.0,2.816327


In [20]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [21]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
112,Adelie,3200.0,3.200,False
18,Adelie,3325.0,3.325,False
312,Gentoo,4750.0,4.750,True
202,Chinstrap,3325.0,3.325,False
288,Gentoo,4700.0,4.700,True
183,Chinstrap,4300.0,4.300,True
178,Chinstrap,3400.0,3.400,False
152,Chinstrap,3500.0,3.500,False
216,Chinstrap,3400.0,3.400,False
197,Chinstrap,4450.0,4.450,True


In [22]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
135,Adelie,3900.0,3.900,8.598018
290,Gentoo,4750.0,4.750,10.471945
137,Adelie,3975.0,3.975,8.763364
324,Gentoo,4725.0,4.725,10.416829
37,Adelie,3550.0,3.550,7.826401
238,Gentoo,4800.0,4.800,10.582176
94,Adelie,3300.0,3.300,7.275246
303,Gentoo,5350.0,5.350,11.794717
311,Gentoo,5400.0,5.400,11.904948
100,Adelie,3725.0,3.725,8.212210


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [23]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
334,Gentoo,False
18,Adelie,True
338,Gentoo,False
302,Gentoo,False
239,Gentoo,False
308,Gentoo,False
195,Chinstrap,False
8,Adelie,True
172,Chinstrap,False
49,Adelie,True


In [24]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
108,Adelie,3175.0,False
21,Adelie,3600.0,False
222,Gentoo,4450.0,True
127,Adelie,4300.0,True
78,Adelie,3550.0,False
45,Adelie,4600.0,True
204,Chinstrap,3600.0,False
114,Adelie,3900.0,False
34,Adelie,3325.0,False
314,Gentoo,4850.0,True


## Overwriting an existing column

In [25]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
174,Chinstrap,2.900
280,Gentoo,4.200
76,Adelie,3.700
25,Adelie,3.800
336,Gentoo,4.875
138,Adelie,3.400
18,Adelie,3.325
302,Gentoo,4.725
153,Chinstrap,3.900
209,Chinstrap,4.050


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [26]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [27]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
29,Adelie,Biscoe,3950.0,40.5,2.408169
9,Adelie,Torgersen,4250.0,42.0,2.409297
295,Gentoo,Biscoe,5800.0,48.6,2.455588
75,Adelie,Torgersen,4250.0,42.8,2.320072
310,Gentoo,Biscoe,4950.0,47.5,2.193906
49,Adelie,Dream,4150.0,42.3,2.319356
114,Adelie,Biscoe,3900.0,39.6,2.486991
43,Adelie,Dream,4400.0,44.1,2.262432
223,Gentoo,Biscoe,5700.0,50.0,2.280000
306,Gentoo,Biscoe,4600.0,43.4,2.442184


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [28]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
153,Chinstrap,3900.0,light
248,Gentoo,4600.0,heavy
87,Adelie,3500.0,light
312,Gentoo,4750.0,heavy
195,Chinstrap,3500.0,light
61,Adelie,4400.0,heavy
48,Adelie,3450.0,light
276,Gentoo,4300.0,heavy
169,Chinstrap,3700.0,light
57,Adelie,3800.0,light


In [29]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
50,Adelie,3500.0,medium
329,Gentoo,5500.0,large
236,Gentoo,4150.0,medium
207,Chinstrap,3450.0,small
269,Gentoo,5300.0,large
341,Gentoo,5750.0,large
132,Adelie,3500.0,medium
241,Gentoo,5000.0,large
331,Gentoo,5950.0,large
90,Adelie,3550.0,medium


In [30]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
191,Chinstrap,Dream,OTH
235,Gentoo,Biscoe,BIS
0,Adelie,Torgersen,TOR
132,Adelie,Dream,OTH
113,Adelie,Biscoe,BIS
140,Adelie,Dream,OTH
180,Chinstrap,Dream,OTH
15,Adelie,Torgersen,TOR
75,Adelie,Torgersen,TOR
242,Gentoo,Biscoe,BIS


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [31]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [32]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`

For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from a file with `#` comments and multiline expressions:

In [33]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
110,Adelie,3.825,8.432671
64,Adelie,2.850,6.283167
228,Gentoo,4.400,9.700328
223,Gentoo,5.700,12.566334
34,Adelie,3.325,7.330361


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [34]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581
